In [1]:
# Import Basic Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import norm, skewnorm
import seaborn as sns
# For EGARCH
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

/Users/akirasohejl/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (
Error importing in API mode: ImportError("dlopen(/Users/akirasohejl/opt/anaconda3/lib/python3.9/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): symbol not found in flat namespace '_R_BaseEnv'")
Trying to import in ABI mode.
/Users/akirasohejl/opt/anaconda3/lib/python3.9/site-packages/rpy2/rinterface/__init__.py:1185: UserWarning: Environment variable "PWD" redefined by R and overriding existing variable. Current: "/Users/akirasohejl", R: "/Users/akirasohejl/Desktop/Humboldt Universität/Master Arbeit/MA_Code/Scripts/notebooks"
  warnings.warn(
/Users/akirasohejl/opt/anaconda3/lib/python3.9/site-packages/rpy2/rinterface/__init__.py:1185: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/fo

In [2]:
def split_series(returns_series, train_ratio=0.8):
    split_idx = int(len(returns_series) * train_ratio)
    train_series = returns_series[:split_idx]
    test_series = returns_series[split_idx:]
    return train_series, test_series

In [3]:
def sim_egarch_returns(n,omega,alpha,beta,gamma,mu,dist = 'normal', seed = 10):
    np.random.seed(seed)
    # Simulate z_t
    if dist == 'normal':
        z = np.random.normal(size=n)
        ez = np.sqrt(2/np.pi)  
    elif dist == 'skewnorm':
        #skew_parm = 5  
        z = skewnorm.rvs(a=skew_parm, size=n)
        ez = np.mean(np.abs(skewnorm.rvs(a=skew_parm, size=100000)))
    else:
        raise ValueError("Unsupported distribution")

    log_sigma2 = np.zeros(n)
    sigma = np.zeros(n)
    returns = np.zeros(n)

    log_sigma2[0] = omega / (1 - beta)
    sigma[0] = np.exp(0.5 * log_sigma2[0])
    returns[0] = mu + sigma[0] * z[0]
    
    for t in range(1, n):
        log_sigma2[t] = omega + beta * log_sigma2[t-1] + gamma * z[t-1] + alpha * (np.abs(z[t-1]) - ez)
        sigma[t] = np.exp(0.5 * log_sigma2[t])
        returns[t] = mu + sigma[t] * z[t]
    return([returns, sigma])

In [4]:
def plot_sim_returns(returns,volatility,gamma, GARCHprocess = "Return"):
    plt.figure(figsize=(10, 4))
    plt.plot(returns, label="Returns")
    plt.plot(volatility, label="Volatility", alpha=0.7)
    plt.title(f"Simulated {GARCHprocess} Process with $\gamma$-Parameter: {gamma}")
    plt.legend()
    plt.tight_layout()
    plt.xlab("Returns")
    plt.ylab("Density")
    plt.show()

In [5]:
def plot_sim_hist(returns,dist,gamma, path = "/Users/akirasohejl/Desktop/Humboldt Universität/Master Arbeit/Code/Plots/Hist",bins=100, xlim = None):
    from scipy.stats import skew
    
    # Compare to normal sample
    normal_returns = np.random.normal(loc=np.mean(returns), scale=np.std(returns), size=len(returns))
    
    # Compute skewness
    skew_normal = skew(normal_returns)
    skew_simulated = skew(returns)

    # Plot histogram comparison
    plt.figure(figsize=(10, 4))
    plt.hist(returns, bins=bins, density=True, alpha=0.6, label=f'Simulated (skew={skew_simulated:.2f})')
    plt.hist(normal_returns, bins=bins, density=True, alpha=0.4, label=f'Normal (skew={skew_normal:.2f})')
    plt.axvline(np.mean(returns), color='black', linestyle='--', label='Mean')
    plt.title(f'Histogram Comparison of Normal vs. Simulated Returns with $\gamma$-Parameter: {gamma}')
    plt.legend()
    plt.tight_layout()
    plt.xlabel("Returns")
    plt.ylabel("Density")
    if xlim != None:
        plt.xlim(xlim)
    plt.savefig(f'{path}/Hist_sim_EGARCH_returns_g{gamma}.png')
    plt.show()

In [6]:
def plot_loss(model_name, loss, sim = False, gamma= None, loss_name = "Loss",path =  "/Users/akirasohejl/Desktop/Humboldt Universität/Master Arbeit/Code/Plots/Loss/"):
    plt.plot(loss, label = "Loss")
    plt.xlabel("Epochs")
    plt.ylabel(f"{loss_name}")
    plt.legend()
    if sim == True:
        plt.title(f"Trainings Loss over Time of {model_name} on simualted returns with $\gamma = {gamma}$")
        plt.savefig(f"{path}/loss_{model_name}_g{gamma}.png")
        
    else:
        plt.title(f"Trainings Loss over Time of {model_name}")
        plt.savefig(f"{path}/loss_{model_name}.png")
    plt.show()

In [8]:
def save_loss(loss_list,model_name, sim= False, path = "/Users/akirasohejl/Desktop/Humboldt Universität/Master Arbeit/Code/Plots/DataApp/Loss/"):
    if sim == True:
        path = "/Users/akirasohejl/Desktop/Humboldt Universität/Master Arbeit/Code/Plots/Loss/"
        pd.Series(loss_list).to_csv(path+model_name+f"_loss_history_g{gamma}.csv", index = False)

    else:
        pd.Series(loss_list).to_csv(path+model_name+"_loss_history.csv", index = False)

In [9]:
def plot_oof_preds(model_name, test_index, gamma,preds, vola_proxy, vola_proxy_name = "Squared Returns", path = "/Users/akirasohejl/Desktop/Humboldt Universität/Master Arbeit/Code/Plots/", sim = True):
    # Plot comparison
    sns.set()
    plt.figure(figsize=(12,6))
    plt.plot(test_index, preds, label=f'{model_name} Predictions', alpha=0.7,linestyle='dashed')
    plt.plot(vola_proxy, label= vola_proxy_name , alpha=0.7, color = 'grey')
    plt.legend()
    plt.xlabel("Time")
    plt.ylabel(f"Variance ($\sigma_t^2$)")
    if sim == True:
        plt.title(f"{model_name} one step-ahead Conditional Variance Forecasts vs. {vola_proxy_name} ($\gamma$ = {gamma})")
        plt.savefig(f'{path}/g{gamma}/{model_name}_sim_preds_g{gamma}.png')
    else:
        plt.title(f"{model_name} one step-ahead Conditional Variance Forecasts vs. {vola_proxy_name}")
        plt.savefig(f'{path}/DataApp/Preds/{model_name}_DA_preds.png')
    plt.show()

In [10]:
def plot_qqs(returns,sim = False,stock_name = None, gamma=None,path = "/Users/akirasohejl/Desktop/Humboldt Universität/Master Arbeit/Code/Plots/QQs"):
    from scipy import stats 
    if sim == True:
        stats.probplot(returns, dist="norm", plot=plt)
        plt.title(f"QQ Plot vs Normal with  $\gamma$ = {gamma}")
        plt.savefig(f'{path}/QQ_plot_sim_returns_g{gamma}.png')
        
    else:
        stats.probplot(returns, dist="norm", plot=plt)
        plt.title(f"QQ Plot vs Normal of {stock_name} Stock returns")
        plt.savefig(f'{path}/QQ_plot_DataApp_returns.png')
    plt.show()

In [11]:
def egarch_fit(train_input, scaling_factor = 0.1,error_dist = "norm"):
    
    returns_clean = train_input * scaling_factor # necessary for convergence

    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv['returns'] = ro.conversion.py2rpy(returns_clean)
    
    r_script = f"""
    library(rugarch)
    set.seed(10)
    spec <- ugarchspec(
        variance.model = list(model = "eGARCH", garchOrder = c(1,1)),
        mean.model = list(armaOrder = c(0,0)),
        distribution.model = "{error_dist}"
    )

    fit <- ugarchfit(spec, data = returns)

    # Capture printed summary as character vector
    summary_output <- capture.output(show(fit))

    # Return the summary
    summary_output
    """
    egarch_summary = ro.r(r_script)
    summary_text = "\n".join(list(egarch_summary))
    print(summary_text)
    return(summary_text)

In [12]:
def egarch_forecast(train_input, test_input, error_dist = "norm",scaling_factor = 0.1, forecast_horizon = 300):
    #import rpy2.robjects as ro
    #from rpy2.robjects import pandas2ri
    #from rpy2.robjects.conversion import localconverter

    train_returns_clean = train_input * scaling_factor # necessary for convergence
    test_returns_clean = test_input * scaling_factor
        
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv['train_returns'] = ro.conversion.py2rpy(train_returns_clean)
        ro.globalenv['test_returns'] = ro.conversion.py2rpy(test_returns_clean)

    r_script = f"""
    library(rugarch)
    full    <- c(train_returns, test_returns)
    n_test  <- length(test_returns)

    spec <- ugarchspec(
      variance.model = list(model="eGARCH", garchOrder=c(1,1)),
      mean.model     = list(armaOrder=c(0,0), include.mean=FALSE),
      distribution.model = "norm"
    )

    # Fit on training only
    fit <- ugarchfit(spec, data=full, out.sample=n_test)

    # 1-step ahead forecast for each test point (origins = last train date + each test date-1)
    fc <- ugarchforecast(fit, n.ahead=1, n.roll=n_test -1)

    # Extract 1-step vol forecasts and convert to variance
    sig1 <- fc@forecast$sigmaFor              # 1 x n_test matrix
    #one_step_var <- as.numeric(sig1)^2
    #names(one_step_var) <- index(test_returns)  # if xts; otherwise use time(test_returns)
    #one_step_var
    sig1"""

    egarch_forecast_r = ro.r(r_script)
    sigma_scaled = pd.Series(list(egarch_forecast_r),index = range(0,len( test_input)))#[1:],index = range(0,len( test_input)))
    
    var_orig   = (sigma_scaled / scaling_factor) ** 2 
    var_orig.index = test_index
    return(var_orig)

In [13]:
def egarch_multistep_forecast(
    train_input, test_input, error_dist="norm", scaling_factor=0.1, forecast_horizon=300
):
    import numpy as np
    import pandas as pd
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.conversion import localconverter

    train_scaled = train_input * scaling_factor
    test_scaled  = test_input  * scaling_factor

    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv["train_returns"] = ro.conversion.py2rpy(train_scaled)
        ro.globalenv["test_returns"]  = ro.conversion.py2rpy(test_scaled)

    ro.globalenv["dist_model"] = ro.StrVector([error_dist])
    ro.globalenv["H"]          = ro.IntVector([int(forecast_horizon)])
    ro.globalenv["n_test"]     = ro.IntVector([int(len(test_input))])

    r_script = """
    suppressMessages(library(rugarch))

    # Build full series and fit only on train (hold out test via out.sample)
    full    <- c(train_returns, test_returns)
    outsmpl <- n_test[1]
    Hh      <- H[1]

    spec <- ugarchspec(
      variance.model = list(model = "eGARCH", garchOrder = c(1,1)),
      mean.model     = list(armaOrder = c(0,0), include.mean = FALSE),
      distribution.model = dist_model[1]
    )

    fit <- ugarchfit(spec, data = full, out.sample = outsmpl)

    # Multi-step forecasts for each origin (last train + rolling through test-1)
    fc <- ugarchforecast(fit, n.ahead = Hh, n.roll = outsmpl )#- 1)

    # sigmaFor: H x (n.roll+1). Convert to variance and transpose => rows=origins
    var_by_origin <- t((fc@forecast$sigmaFor)^2)

    # Nice column names matching arch's h.xxx
    colnames(var_by_origin) <- sprintf("h.%03d", seq_len(ncol(var_by_origin)))

    # Return as data.frame to avoid vector simplification in rpy2
    as.data.frame(var_by_origin)
    """

    with localconverter(ro.default_converter + pandas2ri.converter):
        df = ro.r(r_script)

    s2 = float(scaling_factor) ** 2
    df = df.astype(float) / s2

    origins = list(test_input.index) #[train_input.index[-1]] + list(test_input.index[:-1])
    df = df.iloc[1:,:]
    df.index = pd.Index(origins, name="origin")

    return df


In [14]:
def egarch_multistep_forecast_target(
    train_input, 
    test_input, 
    error_dist="norm", 
    scaling_factor=0.1, 
    forecast_horizon=300
):

    import numpy as np
    import pandas as pd
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.conversion import localconverter

    H = int(forecast_horizon)
    n_test = int(len(test_input))
    if n_test == 0:
        return pd.DataFrame(index=test_input.index)

    train_scaled = train_input * scaling_factor
    test_scaled  = test_input  * scaling_factor

    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv["train_returns"] = ro.conversion.py2rpy(train_scaled)
        ro.globalenv["test_returns"]  = ro.conversion.py2rpy(test_scaled)
    ro.globalenv["dist_model"] = ro.StrVector([error_dist])
    ro.globalenv["Hh"]         = ro.IntVector([H])
    ro.globalenv["outsmpl"]    = ro.IntVector([n_test])

    r_script = """
    suppressMessages(library(rugarch))
    full <- c(train_returns, test_returns)

    spec <- ugarchspec(
      variance.model = list(model = "eGARCH", garchOrder = c(1,1)),
      mean.model     = list(armaOrder = c(0,0), include.mean = FALSE),
      distribution.model = dist_model[1]
    )

    fit <- ugarchfit(spec, data = full, out.sample = outsmpl[1])

    # H x (n.roll+1) with columns = origins, rows = horizons
    # Choose n.roll = n_test - 1 so we get exactly n_test origins:
    # { last-train origin, test_1 origin, ..., test_{n_test-1} origin }
    fc <- ugarchforecast(fit, n.ahead = Hh[1], n.roll = outsmpl[1] - 1)

    # Convert to variance and transpose => rows = origins, cols = horizons
    var_by_origin <- t((fc@forecast$sigmaFor)^2)

    # Nice column names like arch-style (zero-padded)
    colnames(var_by_origin) <- sprintf("h.%03d", seq_len(ncol(var_by_origin)))

    as.data.frame(var_by_origin)
    """

    with localconverter(ro.default_converter + pandas2ri.converter):
        df_origin = ro.r(r_script)

    df_origin = df_origin.astype(float) / (float(scaling_factor) ** 2)

    targets = test_input.index
    H_avail = min(H, df_origin.shape[1])  # safety

    data = {}
    for i in range(1, H_avail + 1):
        col = f"h.{i:03d}"
        s_origin = df_origin[col].copy()

        s_origin.index = targets               
        s_target = s_origin.shift(i - 1)
        data[col] = s_target

    df_target = pd.DataFrame(data, index=targets)

    return df_target


In [15]:
def gaussian_nll(y_true, y_pred):
    #eps = 1e-8  # numerical stability
    sigma2 = y_pred #+ eps
    return 0.5 * (tf.math.log(sigma2) + tf.square(y_true) / sigma2)

In [16]:
def inverse_scale_model_output(pred, scaler, kind="return"):
    scale = scaler.scale_[0]
    mean = scaler.mean_[0]

    if isinstance(pred, torch.Tensor):
        pred = pred.detach().cpu().numpy()

    if kind == "return":
        return pred * scale + mean
    elif kind == "variance":
        return pred * (scale ** 2)
    elif kind == "volatility":
        return pred * scale
    else:
        raise ValueError(f"Unsupported kind '{kind}'. Use 'return', 'variance', or 'volatility'.")

In [17]:
def sequential_preprocess(train_returns, test_returns, reshape_dim = 2):
    from sklearn.preprocessing import MinMaxScaler, StandardScaler
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    
    if reshape_dim == 2:
        X_train_denoise = np.array(train_returns[:-1]).reshape(-1, 1)   
        X_test_denoise = np.array(test_returns[:-1]).reshape(-1, 1)  
        
        y_train_denoise = np.array(train_returns[1:]).reshape(-1, 1) 
        y_test_denoise = np.array(test_returns[1:]).reshape(-1, 1)
        
        X_train_denoised_scaled = scaler_x.fit_transform(X_train_denoise)
        y_train_denoised_scaled = scaler_y.fit_transform(y_train_denoise)#fit_transform(y_train_denoise)


        X_test_denoised_scaled = scaler_x.transform(X_test_denoise) #fit_transform(X_test_denoise)
        y_test_denoised_scaled = scaler_y.transform(y_test_denoise)#fit_transform(y_test_denoise)
    
    if reshape_dim == 3:
        X_train_denoise = np.array(train_returns[:-1]).reshape(-1, 1, 1)
        X_test_denoise = np.array(test_returns[:-1]).reshape(-1, 1, 1)
        
        y_train_denoise = np.array(train_returns[1:]).reshape(-1, 1)
        y_test_denoise = np.array(test_returns[1:]).reshape(-1, 1)

        X_train_denoised_scaled = scaler_x.fit_transform(X_train_denoise.reshape(-1, 1)).reshape(-1, 1, 1)
        X_test_denoised_scaled = scaler_x.transform(X_test_denoise.reshape(-1, 1)).reshape(-1, 1, 1)
        
        y_train_denoised_scaled = scaler_y.fit_transform(y_train_denoise)
        y_test_denoised_scaled = scaler_y.transform(y_test_denoise)
        

    return(X_train_denoised_scaled,y_train_denoised_scaled, X_test_denoised_scaled, y_test_denoised_scaled,scaler_x,scaler_y)

In [18]:
def train_test_split(return_input, split_ratio, scaling = False, scaler = "Standard",reshape_dim = 2):
    from sklearn.preprocessing import MinMaxScaler, StandardScaler
    split = int(len(return_input)*split_ratio)  # train/test split index

    X = (np.array(return_input) ** 2)#[:-1]) ** 2)#*100
    y = (np.array(return_input[1:]) ** 2)#*100

    # Train/test split
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]
    
    if reshape_dim == 2:
        X_train = X_train.reshape(-1, 1)
        X_test = X_test.reshape(-1, 1)
        y_train = y_train.reshape(-1, 1)
        y_test = y_test.reshape(-1, 1)
        
    if reshape_dim == 3:
        X_train = X_train.reshape(-1, 10, 1)
        X_test = X_test.reshape(-1, 10, 1)
        y_train = y_train.reshape(-1, 1)
        y_test = y_test.reshape(-1, 1)
    
    if scaling:    
        if scaler == "Standard":
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
        
        else:
            scaler_x = MinMaxScaler()
            scaler_y = MinMaxScaler()
            
        if reshape_dim == 2:    
            X_train = scaler_x.fit_transform(X_train)
            X_test = scaler_x.transform(X_test)
            y_train = scaler_y.fit_transform(y_train)
            y_test = scaler_y.transform(y_test)
        
        if reshape_dim == 3:
            X_train = scaler_x.fit_transform(X_train.reshape(-1, 1)).reshape(-1, 1, 1)
            X_test = scaler_x.transform(X_test.reshape(-1, 1)).reshape(-1, 1, 1)
            y_train = scaler_y.fit_transform(y_train)
            y_test = scaler_y.transform(y_test)
        
        return(X_train,X_test,y_train,y_test, scaler_x, scaler_y)
    
    else: return(X_train,X_test,y_train,y_test)

In [19]:
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

def make_windows(arr, win):
    X = []
    y = []
    for i in range(len(arr) - win):
        X.append(arr[i:i+win])
        y.append(arr[i+win])
    X = np.array(X)              
    y = np.array(y)             
    return X[..., None], y[..., None] 

def train_test_split_lstm(return_input, split_ratio, win=10, scaling=False, scaler="Standard"):
    r2 = np.array(return_input, dtype=float)**2 
    X, y = make_windows(r2, win)                
    split = int(len(X) * split_ratio)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    if scaling:
        if scaler == "Standard":
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
        else:
            scaler_x = MinMaxScaler()
            scaler_y = MinMaxScaler()

        n_tr, w, f = X_train.shape
        X_train_scaled = scaler_x.fit_transform(X_train.reshape(n_tr * w, f)).reshape(n_tr, w, f)
        n_te = X_test.shape[0]
        X_test_scaled  = scaler_x.transform(X_test.reshape(n_te * w, f)).reshape(n_te, w, f)

        y_train_scaled = scaler_y.fit_transform(y_train)
        y_test_scaled  = scaler_y.transform(y_test)

        return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled, scaler_x, scaler_y

    return X_train, X_test, y_train, y_test


In [20]:
def forecast_metrics_loop(y_test_input, preds_input, model_name, h = 300,round_decimal = 4):
    y_tests = {}
    preds = {}
    rmses = {}
    maes = {}
    mapes = {}
    qlikes = {}
    for i in range(1,h):#[1,5,20,300]:    
        y_test = y_test_input.iloc[:i]
        preds = preds_input[:i]

        rmses[i] = np.round(np.sqrt(np.mean((y_test - preds)**2)),round_decimal)
        maes[i] = np.round(np.mean(np.abs(y_test - preds)),round_decimal)
        mapes[i] = np.round(np.mean(np.abs((y_test - preds) / y_test)) * 100,round_decimal)
        qlikes[i] = np.round(np.mean(np.log(preds) + (y_test) / (preds)),round_decimal)
        print(f"Forecast metrics for forecast horizon h={i}:")
        print("RMSE:",rmses[i])
        print("MAE:",maes[i])
        print("MAPE:",mapes[i])
        print("QLIKE:",qlikes[i])

    df_metrics= pd.concat([
                pd.DataFrame(rmses, index = ["RMSE"]),
                pd.DataFrame(maes, index = ["MAE"]),
                pd.DataFrame(mapes, index = ["MAPE"]),
                pd.DataFrame(qlikes, index = ["QLIKE"]),
                ])
    df_metrics = df_metrics.rename_axis(model_name)#.rename_axis("Forecast Horizon", axis=1)
    return(df_metrics)

In [21]:
def forecast_metrics(y_test, preds, model, round_decimal = 4, qlike_only = False):
    eps = 1e-8
    if  eps >  np.max(preds):
        print("Predictions have lower Maximum-Value than", eps, " and will be replaced for metrics calculation.")
        preds = np.maximum(preds, eps)
    if  eps >  np.max(y_test):
        print("Training Data has lower Maximum-Value than", eps, " and will be replaced for metrics calculation.")
        y_test = np.maximum(y_test, eps)
        
    if np.min(preds) <= 0:
        print("Volatility-Predictions include negative values and will be replace with", eps)
        preds = np.minimum(preds, eps)
        
        
    if qlike_only == True:
        qlike = np.round(np.mean(np.log(preds) + (y_test) / (preds)),round_decimal)
        results = qlike
        #print(f"{model} QLIKE Loss: {qlike:.4f}")
        
    else:
        rmse = np.round(np.sqrt(mean_squared_error(y_test, preds)),round_decimal)
        mae = np.round(mean_absolute_error(y_test, preds),round_decimal)
        mape = np.round(np.mean(np.abs((y_test - preds) / y_test)) * 100,round_decimal)
        qlike = np.round(np.mean(np.log(preds) + (y_test) / (preds)),round_decimal)
        
        results = [model,rmse,mae,mape,qlike]

        # Display results
        #print(f"\nEvaluation Metrics on {model} Volatility:")
        #print(f"RMSE       : {rmse:.4f}")
        #print(f"MAE        : {mae:.4f}")
        #print(f"MAPE (%)   : {mape:.2f}")
        #print(f"QLIKE Loss : {qlike:.4f}")
    return(results)

In [22]:
def plot_acc(returns, lags = 10, path = "/Users/akirasohejl/Desktop/Humboldt Universität/Master Arbeit/Code/Plots/DataApp"):
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    
    for i in range(1,3):
        # Plot ACF
        fig, ax = plt.subplots(figsize=(10, 5))
        plot_acf(returns**i, lags=lags, ax=ax) 
        plt.ylim(-0.1,0.25)
        ax.set_xlabel("Lag")
        ax.set_ylabel("Autocorrelation")
        ax.set_title("Autocorrelation Function (ACF)")
        plt.savefig(path+f"/ACF_returns^{i}.png")
        plt.show()

        # Plot the PACF
        fig, ax = plt.subplots(figsize=(10, 5))
        plot_pacf(returns**i, lags=lags, ax=ax, method='ywm')
        plt.ylim(-0.1,0.25)
        ax.set_xlabel("Lag")
        ax.set_ylabel("Partial Autocorrelation")
        ax.set_title("Partial Autocorrelation Function (PACF)")
        plt.savefig(path+f"/PACF_returns^{i}.png")
        plt.show()

In [23]:
def nn_multi_step_preds_old(model,test_returns, X_test, scaler_y,H,model_name, scaling = True, dim = 2):
    #eps = 1e-12
    N = len(test_returns)
    preds_lt = np.full((N, H), np.nan, dtype=np.float64)
    
    if dim == 3:
        x_scaled_full = X_test.reshape(-1, 1).astype(np.float32)
    
    else:
        x_scaled_full = X_test.reshape(-1, 1, 1).astype(np.float32)

    x_scaled = x_scaled_full 
    for h in range(1, H + 1):
        count = N - h
        if count <= 0:
            break
        yhat_scaled = model.predict(x_scaled[:count], verbose=0)  
        yhat_scaled[~np.isfinite(yhat_scaled)] = 0.0
        
        if scaling == True:
            
            yhat = scaler_y.inverse_transform(yhat_scaled).ravel()
        
        else:
            yhat = yhat_scaled.ravel()
        preds_lt[:count, h - 1] = yhat #np.clip(yhat, eps, None)

        x_scaled = yhat_scaled
        print("Forecast Horizon:",h)

    cols = [f"h.{k:03d}" for k in range(1, H + 1)]
    mlp_preds_origin = pd.DataFrame(preds_lt, index=test_returns.index, columns=cols)
    print(mlp_preds_origin)
    n_step_qlikes_mlp_nll = []
    for i in range(299):
        df_h_step = pd.concat([mlp_preds_origin.iloc[:,i].dropna(),test_returns**2],axis=1).dropna()
        
        n_step_qlikes_mlp_nll.append(forecast_metrics(preds = df_h_step.iloc[:,0],
                         y_test = df_h_step.iloc[:,1],
                         model = model_name,
                         qlike_only = True))#[-1])
    df_qlikes = pd.DataFrame(n_step_qlikes_mlp_nll)
    df_qlikes.to_csv(path + f"/Plots/Loss/{model_name}_loss_g{gamma}.csv",header = False, index = False)
    return(df_qlikes)

In [24]:
def nn_multi_step_preds(model, test_returns, X_test, scaler_y, H, model_name,
                        scaler_x=None):
    N = len(test_returns)
    preds_lt = np.full((N, H), np.nan, dtype=np.float64)

    x_scaled = np.asarray(X_test, dtype=np.float32).reshape(-1, 1)

    if scaler_x is None:
        scaler_x = scaler_y

    for h in range(1, H + 1):
        count = N - h
        if count <= 0:
            break

        x_batch = np.asarray(x_scaled[:count], dtype=np.float32).reshape(count, 1)
        yhat_scaled = model.predict(x_batch, verbose=0).reshape(-1, 1)  # (count,1)
        #yhat_scaled[~np.isfinite(yhat_scaled)] = 0.0
        
        yhat_scaled = np.asarray(yhat_scaled, dtype=np.float64)
        yhat_scaled = np.nan_to_num(yhat_scaled, nan=0.0, posinf=6, neginf=-6)
        yhat_scaled = np.clip(yhat_scaled, -6, 6)

        yhat = scaler_y.inverse_transform(yhat_scaled).ravel().astype(np.float64)
        preds_lt[:count, h - 1] = yhat
        
        next_x_scaled = scaler_x.transform(
            scaler_y.inverse_transform(yhat_scaled)
        ).astype(np.float32).reshape(-1, 1)

        x_scaled = next_x_scaled
        
        print("Forecast Horizon:",h)
    cols = [f"h.{k:03d}" for k in range(1, H + 1)]
    mlp_preds_origin = pd.DataFrame(preds_lt, index=test_returns.index, columns=cols)
    n_step_qlikes = []
    max_h = min(H, preds_lt.shape[1])
    for i in range(max_h):
        df_h = pd.concat([mlp_preds_origin.iloc[:, i], (test_returns ** 2)], axis=1).dropna()
        if df_h.empty:
            break
        n_step_qlikes.append(
            forecast_metrics(preds=df_h.iloc[:, 0],
                             y_test=df_h.iloc[:, 1],
                             model=model_name)[-1]
        )

    df_qlikes = pd.DataFrame(n_step_qlikes)
    df_qlikes.to_csv(path + f"/Plots/Loss/{model_name}_loss_g{gamma}.csv", header=False, index=False)
    return df_qlikes, mlp_preds_origin


In [25]:
def nn_multi_step_preds(model, test_returns, X_test, scaler_y, H, model_name,
                        scaler_x=None, lower_triangular= False):
    N = len(test_returns)
    preds_lt = np.full((N, H), np.nan, dtype=np.float64)

    x_scaled = np.asarray(X_test, dtype=np.float32).reshape(-1, 1)

    if scaler_x is None:
        scaler_x = scaler_y

    for h in range(1, H + 1):
        if lower_triangular == True:
            count = N - h
            
        else:
            count = N
        if count <= 0:
            break

        x_batch = np.asarray(x_scaled[:count], dtype=np.float32).reshape(count, 1)
        yhat_scaled = model.predict(x_batch, verbose=0).reshape(-1, 1)  
        #yhat_scaled[~np.isfinite(yhat_scaled)] = 0.0
        
        yhat_scaled = np.asarray(yhat_scaled, dtype=np.float64)
        yhat_scaled = np.nan_to_num(yhat_scaled, nan=0.0, posinf=6, neginf=-6)
        yhat_scaled = np.clip(yhat_scaled, -6, 6)

        yhat = scaler_y.inverse_transform(yhat_scaled).ravel().astype(np.float64)
        preds_lt[:count, h - 1] = yhat

        next_x_scaled = scaler_x.transform(
            scaler_y.inverse_transform(yhat_scaled)
        ).astype(np.float32).reshape(-1, 1)
        x_scaled = next_x_scaled
        
        print("Forecast Horizon:",h)

    cols = [f"h.{k:03d}" for k in range(1, H + 1)]
    mlp_preds_origin = pd.DataFrame(preds_lt, index=test_returns.index, columns=cols)

    n_step_qlikes = []
    max_h = min(H, preds_lt.shape[1])
    for i in range(max_h):
        df_h = pd.concat([mlp_preds_origin.iloc[:, i], (test_returns ** 2)], axis=1).dropna()
        if df_h.empty:
            break
        n_step_qlikes.append(
            forecast_metrics(preds=df_h.iloc[:, 0],
                             y_test=df_h.iloc[:, 1],
                             model=model_name)[-1]
        )

    df_qlikes = pd.DataFrame(n_step_qlikes)
    df_qlikes.to_csv(path + f"/Plots/Loss/{model_name}_loss_g{gamma}.csv", header=False, index=False)
    return df_qlikes, mlp_preds_origin

In [26]:
# Multi-Step Evaluation
def multi_step_eval(garch_preds, test_returns, H = 300, round_decimal = 2):

    diag_preds, diag_test_sq_returns, diag_df = {}, {}, {}
    diag_qlike, diag_rmse,diag_mae,diag_mape = [],[],[],[]
    
    test_index = test_returns.index 
    
    for h in range(1, H+1):
        col = f"h.{h:03d}"
        if col not in garch_preds.columns:
            break
        diag_preds[h] = garch_preds[col].shift(h).loc[test_index]#.shift(h).loc[test_index]
        diag_df[h] = pd.concat([diag_preds[h], test_returns**2], axis=1).dropna()
        diag_df[h].columns = [col,"sq_returns"]


        preds = diag_df[h][col]
        y_test =diag_df[h].sq_returns
        
        diag_test_sq_returns[h] = test_returns.iloc[h:]**2
        diag_rmse.append(np.sqrt(np.mean((preds - y_test)**2)) )
        diag_mae.append(np.mean(np.abs(y_test - preds)))
        diag_mape.append(np.mean(np.abs((y_test - preds) / y_test)) * 100)
        diag_qlike.append(np.mean((y_test/preds) + np.log(preds)))
    diag_metrics = pd.DataFrame(np.round([diag_rmse,diag_mae,diag_mape,diag_qlike],round_decimal)).T
    diag_metrics.columns = ["RMSE","MAE","MAPE","QLIKE"]

    return(diag_df, diag_metrics)

In [27]:
def plot_n_step_preds(preds, model_name_short, 
                      model_name_long, 
                      h_steps = [1,5,10,50], 
                      uncond_var = None, ylim = None):
    plt.figure(figsize=(12,8))
    plt.title(f"{model_name_long}: $h$-step ahead Conditional Variance Forecasts ($\gamma = {gamma}$)")

    
    if uncond_var != None:
        plt.title(f"{model_name_long}: $h$-step Conditional Variance Forecast vs. Unconditional Variance Estimate $V[r_t]$ for $\gamma = {gamma}$")
        plt.plot(test_index,np.repeat(uncond_var,300), 
                 label = "$V[r_t]=$"  + str(uncond_var), 
                 color = "grey",
                 linestyle = "dashed")
        
    for i in h_steps:
        plt.plot(preds.loc[1200 + i:,f"h.{i:03d}"], label = f"$h = {i}$", alpha =0.8 - (i  /100))
    plt.xlabel("Time")
    plt.ylabel("Conditional Variance $\sigma^2_t$")
    
    if ylim != None:
        plt.ylim(ylim)
    plt.legend()
    plt.savefig(path + f"Plots/Econ/{model_name_short}_uncond_var_g{gamma}.png")

In [28]:
def multi_step_preds_torch(model,model_name, test_returns,H=300, path = '/Users/akirasohejl/Desktop/Humboldt Universität/Master Arbeit/Code/',lower_triangluar=False):

    N = len(test_returns)
    X_test = test_returns.array.reshape(-1, 1)

    # Build origin vector r_t 
    r_test = torch.tensor(test_returns.values, dtype=torch.float32).view(-1, 1)

    r = r_test.clone()                
    idx = list(test_returns.index)

    N = r.shape[0]
    r2 = r.pow(2)

    preds = np.full((N, H), np.nan, dtype=np.float64)
    x_scaled_full = X_test.reshape(-1, 1).astype(np.float32)
    x_scaled = x_scaled_full  
    model.eval()

    for h in range(1, H + 1):
        
            if lower_triangluar == True:
                count = N - h
            
            else: 
                count = N
            if count <= 0:
                break

            r_in  = r[:count].view(-1)
            r2_in = r2[:count].view(-1)

            v_hat = model(r_in, r2_in)          
            v_hat = v_hat.view(-1, 1)                     
            v_np  = v_hat.squeeze(-1).detach().cpu().numpy()

            store = np.maximum(v_np, 0.0)
            preds[:count, h - 1] = store 
            r  = torch.zeros_like(v_hat)               
            r2 = v_hat

            print("Forecast Horizon:",h)

    cols = [f"h.{k:03d}" for k in range(1, H + 1)]
    n_step_preds_matrix = pd.DataFrame(preds, index=test_returns.index, columns=cols)
    
    n_step_qlikes = []
    for i in range(H):#299):
        df_h_step = pd.concat([n_step_preds_matrix.iloc[:,i].dropna(),test_returns**2],axis=1).dropna()

        n_step_qlikes.append(forecast_metrics(preds = df_h_step.iloc[:,0],
                         y_test = df_h_step.iloc[:,1],
                         model = model_name,
                         qlike_only = True))#[-1])
    df_qlikes = pd.DataFrame(n_step_qlikes)
    df_qlikes.to_csv(path + f"/Plots/Loss/{model_name}_loss_g{gamma}.csv",header = False, index = False)
    return(df_qlikes,n_step_preds_matrix)

In [29]:
def lstm_multi_step_preds(model,
                          test_returns,          
                          X_test_seq,            
                          scaler_x,              
                          scaler_y,              
                          H,
                          model_name,
                          path,
                          gamma,
                         lower_triangular = False):

    N, seq_len, feat = X_test_seq.shape
    assert feat == 1, "X_test_seq must have last dimension = 1 (one feature)."
    preds_lt = np.full((N, H), np.nan, dtype=np.float64)

    seqs = X_test_seq.astype(np.float32).copy()

    for h in range(1, H + 1):
        if lower_triangular == True:
            count = N - h
            
        else:
            count = N
        if count <= 0:
            break

        yhat_scaled = model.predict(seqs[:count], verbose=0).reshape(-1, 1)   
        yhat = scaler_y.inverse_transform(yhat_scaled).ravel()                 
        preds_lt[:count, h - 1] = yhat

        yhat_for_x = scaler_x.transform(yhat.reshape(-1, 1)).reshape(-1, 1, 1)
        seqs = np.concatenate([seqs[:, 1:, :], np.zeros((N, 1, 1), dtype=np.float32)], axis=1)
        seqs[:count, -1:, :] = yhat_for_x                            
        
        print("Forecast:",h)
        
    cols = [f"h.{k:03d}" for k in range(1, H + 1)]
    #print(pd.DataFrame(preds_lt))
    lstm_preds_origin = pd.DataFrame(preds_lt, index=test_returns.index[1:], columns=cols)
    n_step_qlikes = []
    max_h = min(H, preds_lt.shape[1])
    for i in range(max_h):
        df_h = pd.concat([lstm_preds_origin.iloc[:, i], (test_returns ** 2)], axis=1).dropna()
        n_step_qlikes.append(
            forecast_metrics(preds=df_h.iloc[:, 0],
                             y_test=df_h.iloc[:, 1],
                             model=model_name,
                             qlike_only = True)#[-1]
        )

    df_qlikes = pd.DataFrame(n_step_qlikes)
    df_qlikes.to_csv(path + f"/Plots/Loss/{model_name}_loss_g{gamma}.csv", header=False, index=False)
    return df_qlikes, lstm_preds_origin

In [30]:
def lstm_multi_step_preds(model,
                          test_returns,          
                          X_test_seq,            
                          scaler_x,              
                          scaler_y,              
                          H,
                          model_name,
                          path,
                          gamma,
                          lower_triangular = False):

    N, seq_len, feat = X_test_seq.shape
    assert feat == 1, "X_test_seq must have last dimension = 1 (one feature)."
    preds_lt = np.full((N, H), np.nan, dtype=np.float64)

    seqs = X_test_seq.astype(np.float32).copy()

    for h in range(1, H + 1):
        
        if lower_triangular == True:
            count = N - h
            
        else:
            count = N
        if count <= 0:
            break

        
        yhat_scaled = model.predict(seqs[:count], verbose=0).reshape(-1, 1)    
        yhat = scaler_y.inverse_transform(yhat_scaled).ravel()                
        preds_lt[:count, h - 1] = yhat

        yhat_for_x = scaler_x.transform(yhat.reshape(-1, 1)).reshape(-1, 1, 1) 
        seqs = np.concatenate([seqs[:, 1:, :], np.zeros((N, 1, 1), dtype=np.float32)], axis=1)
        seqs[:count, -1:, :] = yhat_for_x                           
        
        print("Forecast:",h)
        
    cols = [f"h.{k:03d}" for k in range(1, H + 1)]
    lstm_preds_origin = pd.DataFrame(preds_lt, index=test_returns.index, columns=cols)
    n_step_qlikes = []
    max_h = min(H, preds_lt.shape[1])
    for i in range(max_h):
        df_h = pd.concat([lstm_preds_origin.iloc[:, i], (test_returns ** 2)], axis=1).dropna()
        n_step_qlikes.append(
            forecast_metrics(preds=df_h.iloc[:, 0],
                             y_test=df_h.iloc[:, 1],
                             model=model_name,
                             qlike_only = True)#[-1]
        )

    df_qlikes = pd.DataFrame(n_step_qlikes)
    #df_qlikes.to_csv(path + f"/Plots/Loss/{model_name}_loss_g{gamma}.csv", header=False, index=False)
    return df_qlikes, lstm_preds_origin

In [31]:
def comp_nstep_qlike_rmse(preds_matrix, test_proxy1, test_proxy2, H, decimal = 2):
    qlikes_rsq = []
    qlikes_rv = []
    rmse_rsq = []
    rmse_rv = []
    
    for i in range(0,100):
        qlikes_rsq.append(forecast_metrics(preds = preds_matrix.iloc[:,i],
                           y_test = test_proxy1, 
                           model = "")[-1])
    
        qlikes_rv.append(forecast_metrics(preds = preds_matrix.iloc[:,i],
                               y_test = test_proxy2, 
                               model = "")[-1])

        rmse_rsq.append(forecast_metrics(preds = preds_matrix.iloc[:,i],
                               y_test = test_proxy1, 
                               model = "")[1])

        rmse_rv.append(forecast_metrics(preds = preds_matrix.iloc[:,i],
                               y_test = test_proxy2, 
                               model = "")[1])
    
    print("(Sq.R. h = 1), (Sq.R. h = 100), (RV h = 1), (RV, h = 100) \n")
    print("QLIKE Losses: \n",
      np.round([qlikes_rsq[0],
                qlikes_rsq[-1],
                  qlikes_rv[0],
                qlikes_rv[-1]],decimal))

    print("RMSE Losses: \n",
          np.round([rmse_rsq[0],
                    rmse_rsq[-1],
                    rmse_rv[0],
                    rmse_rv[-1]],decimal))
        
    return(qlikes_rsq,qlikes_rv,rmse_rsq,rmse_rv )

In [32]:
from statsmodels.tsa.stattools import adfuller, kpss
def run_adf(series, lags):#autolag='AIC'):
   
    result = adfuller(series, maxlag=lags,autolag=None)#autolag=autolag)
    output = {
        'ADF Statistic': result[0],
        'p-value': result[1],
        'Lags used': result[2],
        'Observations': result[3],
        'Critical values': result[4]
    }
    return output


def run_kpss_test(series, regression='c', nlags='auto'):
    """
    KPSS test
    H0: Series is stationary
    H1: Series is non-stationary
    regression='c' → level stationarity
    regression='ct' → trend stationarity
    """
    statistic, p_value, lags, crit_vals = kpss(series, regression=regression, nlags=nlags)
    output = {
        'KPSS Statistic': statistic,
        'p-value': p_value,
        'Lags used': lags,
        'Critical values': crit_vals
    }
    return output

In [33]:
import statsmodels.api as sm

def engle_ng_sign_bias(y):

    y = np.asarray(y)
    sigma2 = np.var(y)
    et = y / np.sqrt(sigma2)

    et_lag_sq = et**2
    I_neg = (et < 0).astype(int)

    X1 = sm.add_constant(I_neg)
    model1 = sm.OLS(et_lag_sq, X1).fit()
    sign_bias_t = model1.tvalues[1]
    sign_bias_p = model1.pvalues[1]

    X2 = sm.add_constant(I_neg * et)
    model2 = sm.OLS(et_lag_sq, X2).fit()
    neg_size_bias_t = model2.tvalues[1]
    neg_size_bias_p = model2.pvalues[1]

    I_pos = (et > 0).astype(int)
    X3 = sm.add_constant(I_pos * et)
    model3 = sm.OLS(et_lag_sq, X3).fit()
    pos_size_bias_t = model3.tvalues[1]
    pos_size_bias_p = model3.pvalues[1]

    results = pd.DataFrame({
        "Test": ["Sign Bias", "Negative Size Bias", "Positive Size Bias"],
        "t-Stat": [sign_bias_t, neg_size_bias_t, pos_size_bias_t],
        "p-Value": [sign_bias_p, neg_size_bias_p, pos_size_bias_p]
    })

    return results